# Prediction for Picnic Weather
## Base Model

Fit base model with processed dataset

Use RandomForestCassifier as first approach

---

## 1. Load Modules & Load Data



In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from core.data import load_from_kaggle

# Styling für bessere Visualisierungen
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# GeoPy with Nominatim to estimate lat, lon from city name
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os
import time
import requests     # um die OpenAPI für Höhenangaben bei lat, lon abzufragen

# Meteostat for weather data
from datetime import date
import meteostat as ms

# Models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib   # um Modelle zu speichern und zu laden
# from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance

# import module for Score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import cross_val_score



## 2. Daten einlesen & erste Inspektion



In [ ]:

file_weather_FS    = '../data/processed/weather_FS_lat_lon_elevation.csv'
file_weather_meteostat    = '../data/processed/weather_meteostat_lat_lon_elevation.csv'

# For extended analysis, you could also load the complete dataset if needed
weather_FS = pd.read_csv(file_weather_FS, 
                           delimiter=',', encoding='ascii', parse_dates=['DATE'])
print(f"📊 Geladene Datei: {file_weather_FS}")
print(f"📏 Shape: {weather_FS.shape[0]:,} Zeilen × {weather_FS.shape[1]} Spalten\n")

# For extended analysis, you could also load the complete dataset if needed
meteostat = pd.read_csv(file_weather_meteostat, 
                           delimiter=',', encoding='ascii', parse_dates=['DATE'])
print(f"📊 Geladene Datei: {file_weather_meteostat}")
print(f"📏 Shape: {meteostat.shape[0]:,} Zeilen × {meteostat.shape[1]} Spalten\n")

# Convert the DATE column to datetime. The DATE is given as an integer (likely in YYYYMMDD format)
# for df in [weather_FS, meteostat]:
#     df['DATE'] = pd.to_datetime(df['DATE'].astype(str), format='%Y%m%d', errors='coerce')

print('Data loaded and DATE column converted to datetime.')

display(weather_FS.sample(3))
meteostat.sample(3)

weather_FS.info()
meteostat.info()




---

## 3. Datenqualität & Struktur



In [ ]:
# Func Def: Overview(df) Übersicht entwerfen, um immer mal wieder einen schnellen Überblick über die wichtigsten "Metadaten" zu bekommen
def overview(df):
    '''
    Erstelle einen Überblick über einige wichtige Eigenschaften der Spalten eines DataFrames.
    VARs
        df: Der zu betrachtende DataFrame
    RETURNS:
        None
    '''
    df = df.copy()
    display(pd.DataFrame({'dtype': df.dtypes,
                          'total': df.count(),
                          'missing_n': df.isna().sum(),
                          'missing_%': df.isna().mean()*100,
                          'uniques_n': df.nunique(),
                          'uniques': [df[col].unique() for col in df.columns]
                         }))

In [ ]:
overview(weather_FS)
overview(meteostat)

In [ ]:
# col_names = ['Date', 'Month',
#              'cloud_cover', 'humidity', 'pressure', 'global_radiation',
#              'precipitation', 'sunshine', 'temp_mean', 'temp_min', 'temp_max',
#              'lat', 'lon', 'city']

# cities = ['BASEL', 'BUDAPEST', 'DE', 'DRESDEN', 'DUSSELDORF', 'HEATHROW', 'KASSEL', 'LJUBLJANA',
#           'MAASTRICHT', 'MALMO', 'MONTELIMAR', 'MUENCHEN', 'OSLO', 'PERPIGNAN',
#           'SONNBLICK', 'STOCKHOLM', 'TOURS']

# df_basel = weather_full.loc[:,['DATE', 'MONTH', 'BASEL_cloud_cover', 'BASEL_humidity', 'BASEL_pressure', 'BASEL_global_radiation',
#                                'BASEL_precipitation', 'BASEL_sunshine', 'BASEL_temp_mean', 'BASEL_temp_min', 'BASEL_temp_max']]
# df_basel['lat'] = 47.5596
# df_basel['lon'] = 7.5886
# df_basel['city'] = 'Basel'
# df_basel.columns = col_names
# df_basel

In [ ]:
weather_FS

In [ ]:
weather_FS.city.unique()



---

## Base Model for Picnic Weather Prediction

In [ ]:
# Function to create, fit and evaluate a model for a CITY with RandomForestClassifier

def fit_model_for_city(city_name, df_weather, feature_cols, target_col, save_plot=False, show_plot=True):

    if city_name == 'EURO':
        X = df_weather[feature_cols]
        y = df_weather[target_col]
    else:
        X = df_weather[df_weather['city'] == city_name][feature_cols]
        y = df_weather[df_weather['city'] == city_name][target_col]

    # overview(X)

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize and train a Random Forest Classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    clf.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = clf.predict(X_test)

    # Calculate prediction accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print('Prediction accuracy for ' + city_name + ' picnic weather:', accuracy)

    # Calculate ROC curve and AUC
    y_prob = clf.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    print('AUC:', roc_auc)

    # Compute permutation importance
    perm_importance = permutation_importance(clf, X_test, y_test, n_repeats=10, random_state=42)
    importances = perm_importance.importances_mean

    # Create a DataFrame for feature importance
    feature_importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
    feature_importance_df.sort_values(by='importance', ascending=True, inplace=True)

    print('Permutation importance calculated for the features.')

    print(f'RandomForestClassifier \t\t\t|\t TEST \t\t|\t TRAIN')
    print(f'-------------------------------------------------------------------------------------')
    print(f'True #IsBad:     \t\t\t|\t {(y_test==1).sum()} \t\t|\t {(y_train==1).sum()}')
    print(f'True #NotBad:    \t\t\t|\t {(y_test==0).sum()} \t\t|\t {(y_train==0).sum()}')
    print(f'F1-Score:        \t\t\t|\t {f1_score(y_test, clf.predict(X_test))*100:.2f} % \t|\t '
                                f'{f1_score(y_train, clf.predict(X_train))*100:.2f} %')
    print(f'precision_score: \t\t\t|\t {precision_score(y_test, clf.predict(X_test))*100:.2f} % \t|\t '
                                f'{precision_score(y_train, clf.predict(X_train))*100:.2f} %')
    print(f'recall_score:    \t\t\t|\t {recall_score(y_test, clf.predict(X_test))*100:.2f} % \t|\t '
                                f'{recall_score(y_train, clf.predict(X_train))*100:.2f} %')
    print(f'accuracy_score:  \t\t\t|\t {accuracy_score(y_test, clf.predict(X_test))*100:.2f} % \t|\t '
                                f'{accuracy_score(y_train, clf.predict(X_train))*100:.2f} %')

    if save_plot:
        os.makedirs('../plots', exist_ok=True)

    # Model Evaluation and Visualization
    # Plot ROC Curve
    roc_fig = plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], '--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for ' + city_name + ' Picnic Weather Prediction')
    plt.legend(loc='lower right')
    if save_plot:
        roc_fig.savefig(f'../plots/ROC_{city_name}.png', bbox_inches='tight')
    if show_plot:
        plt.show()
    else:
        plt.close(roc_fig)

    # Plot Confusion Matrix using a heatmap
    cm = confusion_matrix(y_test, y_pred)
    cm_fig = plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix ' + city_name)
    if save_plot:
        cm_fig.savefig(f'../plots/Confusion_{city_name}.png', bbox_inches='tight')
    if show_plot:
        plt.show()
    else:
        plt.close(cm_fig)

    # Plot Permutation Importance
    imp_fig = plt.figure(figsize=(8, 6))
    plt.barh(feature_importance_df['feature'], feature_importance_df['importance'], color='skyblue')
    plt.xlabel('Mean Importance')
    plt.title('Permutation Importance of ' + city_name + ' Weather Features')
    if save_plot:
        imp_fig.savefig(f'../plots/Importance_{city_name}.png', bbox_inches='tight')
    if show_plot:
        plt.show()
    else:
        plt.close(imp_fig)



In [ ]:
# Function to create, fit and evaluate a model for a CITY with XGBoost
def fit_model_xgb(city_name, df_weather, feature_cols, target_col, save_plot=False, show_plot=True, save_model=False):

    if city_name == 'EURO':
        X = df_weather[feature_cols]
        y = df_weather[target_col]
    else:
        X = df_weather[df_weather['city'] == city_name][feature_cols]
        y = df_weather[df_weather['city'] == city_name][target_col]

    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    # overview(X)

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize and train an XGBoost classifier
    clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    clf.fit(X_train, y_train)

    # Save the trained model to a joblib file
    if save_model:
        model_path = f'model_xgb_{city_name}.joblib'
        joblib.dump(clf, model_path)
        print(f'Model saved to: {model_path}')

    # Make predictions on the test set
    y_pred = clf.predict(X_test)

    # Calculate prediction accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print('Prediction accuracy for ' + city_name + ' picnic weather with XGBoost:', accuracy)

    # Calculate ROC curve and AUC
    y_prob = clf.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    print('AUC:', roc_auc)

    # Compute permutation importance
    perm_importance = permutation_importance(clf, X_test, y_test, n_repeats=10, random_state=42)
    importances = perm_importance.importances_mean

    # Create a DataFrame for feature importance
    feature_importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
    feature_importance_df.sort_values(by='importance', ascending=True, inplace=True)

    print('Permutation importance calculated for the features.')

    print(f'XGBClassifier \t\t\t|\t TEST \t\t|\t TRAIN')
    print(f'-------------------------------------------------------------------------------------')
    print(f'True #IsBad:     \t\t\t|\t {(y_test==1).sum()} \t\t|\t {(y_train==1).sum()}')
    print(f'True #NotBad:    \t\t\t|\t {(y_test==0).sum()} \t\t|\t {(y_train==0).sum()}')
    print(f'F1-Score:        \t\t\t|\t {f1_score(y_test, clf.predict(X_test))*100:.2f} % \t|\t '
                                f'{f1_score(y_train, clf.predict(X_train))*100:.2f} %')
    print(f'precision_score: \t\t\t|\t {precision_score(y_test, clf.predict(X_test))*100:.2f} % \t|\t '
                                f'{precision_score(y_train, clf.predict(X_train))*100:.2f} %')
    print(f'recall_score:    \t\t\t|\t {recall_score(y_test, clf.predict(X_test))*100:.2f} % \t|\t '
                                f'{recall_score(y_train, clf.predict(X_train))*100:.2f} %')
    print(f'accuracy_score:  \t\t\t|\t {accuracy_score(y_test, clf.predict(X_test))*100:.2f} % \t|\t '
                                f'{accuracy_score(y_train, clf.predict(X_train))*100:.2f} %')

    if save_plot:
        os.makedirs('../plots', exist_ok=True)

    # Model Evaluation and Visualization
    roc_fig = plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], '--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for ' + city_name + ' Picnic Weather Prediction (XGBoost)')
    plt.legend(loc='lower right')
    if save_plot:
        roc_fig.savefig(f'../plots/ROC_XGB_{city_name}.png', bbox_inches='tight')
    if show_plot:
        plt.show()
    else:
        plt.close(roc_fig)

    cm = confusion_matrix(y_test, y_pred)
    cm_fig = plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix ' + city_name + ' (XGBoost)')
    if save_plot:
        cm_fig.savefig(f'../plots/Confusion_XGB_{city_name}.png', bbox_inches='tight')
    if show_plot:
        plt.show()
    else:
        plt.close(cm_fig)

    imp_fig = plt.figure(figsize=(8, 6))
    plt.barh(feature_importance_df['feature'], feature_importance_df['importance'], color='skyblue')
    plt.xlabel('Mean Importance')
    plt.title('Permutation Importance of ' + city_name + ' Weather Features (XGBoost)')
    if save_plot:
        imp_fig.savefig(f'../plots/Importance_XGB_{city_name}.png', bbox_inches='tight')
        plt.close(imp_fig)

    if show_plot:
        plt.show()

In [ ]:
staedte_features = {
    # 'BASEL': ['BASEL_cloud_cover', 'BASEL_humidity', 'BASEL_pressure', 'BASEL_global_radiation',
    #           'BASEL_precipitation', 'BASEL_sunshine', 'BASEL_temp_mean', 'BASEL_temp_min', 'BASEL_temp_max'],
    'BASEL': ['BASEL_cloud_cover', 'BASEL_humidity', 'BASEL_pressure', 'BASEL_global_radiation',
              'BASEL_temp_mean', 'BASEL_temp_min', 'BASEL_temp_max'],
    'DE_BILT': ['DE_BILT_cloud_cover', 'DE_BILT_humidity', 'DE_BILT_pressure', 'DE_BILT_global_radiation',
                'DE_BILT_precipitation', 'DE_BILT_sunshine', 'DE_BILT_temp_mean',
                'DE_BILT_temp_min', 'DE_BILT_temp_max'],
    'DRESDEN': ['DRESDEN_cloud_cover', 'DRESDEN_humidity', 'DRESDEN_global_radiation',
                'DRESDEN_precipitation', 'DRESDEN_sunshine', 'DRESDEN_temp_mean',
                'DRESDEN_temp_min', 'DRESDEN_temp_max'],
    'DUSSELDORF': ['DUSSELDORF_cloud_cover', 'DUSSELDORF_humidity', 'DUSSELDORF_pressure',
                   'DUSSELDORF_global_radiation', 'DUSSELDORF_precipitation', 'DUSSELDORF_sunshine',
                   'DUSSELDORF_temp_mean', 'DUSSELDORF_temp_min', 'DUSSELDORF_temp_max'],
    'HEATHROW': ['HEATHROW_cloud_cover', 'HEATHROW_humidity', 'HEATHROW_pressure', 'HEATHROW_global_radiation',
                 'HEATHROW_precipitation', 'HEATHROW_sunshine', 'HEATHROW_temp_mean',
                 'HEATHROW_temp_min', 'HEATHROW_temp_max'],
    'KASSEL': ['KASSEL_humidity', 'KASSEL_pressure', 'KASSEL_global_radiation', 'KASSEL_precipitation',
               'KASSEL_sunshine', 'KASSEL_temp_mean', 'KASSEL_temp_min', 'KASSEL_temp_max'],
    'MAASTRICHT': ['MAASTRICHT_cloud_cover', 'MAASTRICHT_humidity', 'MAASTRICHT_pressure', 'MAASTRICHT_global_radiation',
                   'MAASTRICHT_precipitation', 'MAASTRICHT_sunshine', 'MAASTRICHT_temp_mean', 'MAASTRICHT_temp_min',
                   'MAASTRICHT_temp_max'],
    'MALMO': ['MALMO_precipitation', 'MALMO_temp_mean', 'MALMO_temp_min', 'MALMO_temp_max'],
    'MUENCHEN': ['MUENCHEN_cloud_cover', 'MUENCHEN_humidity', 'MUENCHEN_pressure', 'MUENCHEN_global_radiation',
                 'MUENCHEN_precipitation', 'MUENCHEN_sunshine', 'MUENCHEN_temp_mean', 'MUENCHEN_temp_min', 'MUENCHEN_temp_max'],
    # 'SONNBLICK': ['SONNBLICK_cloud_cover', 'SONNBLICK_humidity', 'SONNBLICK_global_radiation', 'SONNBLICK_precipitation',
    #               'SONNBLICK_sunshine', 'SONNBLICK_temp_mean', 'SONNBLICK_temp_min', 'SONNBLICK_temp_max'],
    'TOURS': ['TOURS_humidity', 'TOURS_pressure', 'TOURS_global_radiation', 'TOURS_precipitation',
              'TOURS_temp_mean', 'TOURS_temp_min', 'TOURS_temp_max'],
}




In [ ]:
europe_features = {
    'EUROall': ['cloud_cover', 'wind_speed', 'wind_gust', 'humidity', 'pressure', 'global_radiation',
              'precipitation', 'sunshine', 'temp_mean', 'temp_min', 'temp_max', 'lat', 'lon', 'elevation'],
    # 'EUROselect': ['MONTH', 'precipitation', 'temp_mean', 'temp_max', 'lat', 'lon', 'elevation'],
    'EUROselect': ['MONTH', 'temp_mean', 'temp_max', 'cloud_cover', 'wind_speed', 'sunshine', 
                   'wind_gust', 'humidity', 'pressure', 'global_radiation', 'lat', 'lon', 'elevation'],
    'EUROmin': ['MONTH', 'temp_mean', 'temp_max', 'lat', 'lon', 'elevation'],
              }

In [ ]:
city_name = 'BASEL'
city_name = 'EURO'
# feature_cols = staedte_features[city_name]
# feature_cols = europe_features['EUROmin']
feature_cols = europe_features['EUROselect']
# fit_model_for_city(city_name, weather_FS, feature_cols,
#                    ['picnic_weather'], save_plot=False, show_plot=True)

fit_model_xgb(city_name, weather_FS, feature_cols,
                   ['picnic_weather'], save_plot=False, show_plot=True, save_model=True)


In [ ]:
fit_model_for_city('EUROmin', weather_FS, weather_FS['picnic_weather'],
                   europe_features['EUROmin'], save_plot=True, show_plot=True)
# fit_model_for_city('EUROselect', weather_FS, weather_FS['picnic_weather'],
#                    europe_features['EUROselect'], save_plot=True, show_plot=True)



In [ ]:
# Beispiel: Zugriff per Stadtname
# city_name = 'TOURS'
# feature_cols = staedte_features[city_name]
# fit_model_for_city(city_name, weather_light, picnic_labels, feature_cols)

unique_cities = list(staedte_features.keys()) # weather_FS['city'].unique()

for city in unique_cities:
    fit_model_for_city(city, weather_light, picnic_labels, 
                       staedte_features[city], save_plot=True, show_plot=False)

## Model Evaluation and Visualization



---

## 4. Numerische Variablen analysieren





---

## 5. Kategorische Variablen analysieren





---

## 6. Korrelationsanalyse





---

## 7. Fehlende Werte visualisieren





---

## 8. Zusammenfassung & Bewertung

